# L07 · Building a Grasping Task Scene

This lab turns the prepared YCB assets and scene configuration into the reusable task world needed by later manipulation lessons:

```text
prepared assets → layout geometry → SceneBundle → settle and measure
→ optional world/wrist observations → L08 task execution
```

The numerical path builds and checks the complete base scene without rendering. The optional camera branch adds real world and wrist RGB/depth observations from the same scene.


## Before you run

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD backend when available and otherwise uses CPU. Set it to `cpu` to require the minimum-compatible backend. `ROBO_GENESIS_RENDER=0` runs the complete numerical scene check without creating cameras; set it to `1` before starting the kernel to require both world and wrist RGB/depth observations. Restart the kernel before changing either setting.

Predict first:

1. Why is each object origin z above `TABLE_TOP_Z`?
2. Why is center-on-table insufficient for validating an object placement?
3. Which `SceneBundle` handles should exist after a non-rendering build?
4. What different scene information should the world and wrist views expose?


In [ ]:
import os
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.build_scene import build_scene
from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.scene_config import (
    FRANKA_QPOS,
    REACH_X,
    REACH_Y,
    TABLE_CENTER,
    TABLE_TOP_SIZE,
    TABLE_TOP_Z,
    WORLD_CAM_RES,
    WRIST_CAM_RES,
    YCB_LAYOUT,
    get_ycb_assets,
)
from robo_genesis.setup_assets import YCB_OBJECTS, setup_assets

lesson = load_course_manifest().lesson("L07")
assert lesson.slug == "building-a-grasping-task-scene"
assert lesson.duration_minutes == 90
assert lesson.status.value == "planned"

backend_mode = os.environ.get("ROBO_GENESIS_BACKEND", "auto").strip().lower()
if backend_mode not in {"auto", "cpu"}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get("ROBO_GENESIS_RENDER", "0").strip()
if render_value not in {"0", "1"}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == "1"

runtime = notebook_mode("l07-grasping-task-scene", show_viewer=False)
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == "cpu" else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision="32", logging_level="warning")

if getattr(gs, "amdgpu", None) is not None and gs.backend == gs.amdgpu:
    actual_backend = "amdgpu"
elif gs.backend == gs.cpu:
    actual_backend = "cpu"
else:
    actual_backend = str(gs.backend)

print("Genesis:", environment["genesis_world"])
print("requested backend:", backend_mode)
print("actual backend:", actual_backend)
print("render enabled:", render_enabled)
print("output directory:", runtime["output_dir"].resolve())


## Resolve the prepared community assets

The project preflight returns the supported YCB models directory. `get_ycb_assets()` then exposes one stable mesh path plus derived rest-height and footprint geometry for each task object. These records are configuration inputs; Genesis entities do not exist yet.


In [ ]:
models_dir = setup_assets()
assets = get_ycb_assets(models_dir)

asset_checks = {
    "supported_object_ids": tuple(assets) == YCB_OBJECTS,
    "mesh_paths_exist": all(asset.mesh_path.is_file() for asset in assets.values()),
    "derived_geometry_finite": all(
        np.isfinite([asset.rest_z_offset, asset.radius_xy]).all()
        and asset.rest_z_offset > 0.0
        and asset.radius_xy > 0.0
        for asset in assets.values()
    ),
}
for name, passed in asset_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(asset_checks.values()):
    raise AssertionError(asset_checks)

print("models directory:", models_dir)
for name, asset in assets.items():
    print(
        f"{name:>11}  mesh={asset.mesh_path}  "
        f"rest_z={asset.rest_z_offset:.6f} m  radius_xy={asset.radius_xy:.6f} m"
    )


## Derive and audit the base layout

For each object, derive `origin_z = TABLE_TOP_Z + rest_z_offset` rather than guessing an origin height. Then treat `radius_xy` as a conservative circular footprint: the center, full footprint, course working-region membership, and every pairwise separation margin are checked independently.


In [ ]:
table_xy_bounds = np.array(
    [
        [TABLE_CENTER[0] - TABLE_TOP_SIZE[0] / 2, TABLE_CENTER[0] + TABLE_TOP_SIZE[0] / 2],
        [TABLE_CENTER[1] - TABLE_TOP_SIZE[1] / 2, TABLE_CENTER[1] + TABLE_TOP_SIZE[1] / 2],
    ],
    dtype=float,
)
working_xy_bounds = np.array([REACH_X, REACH_Y], dtype=float)


def inside_xy(point, bounds, margin=0.0):
    point = np.asarray(point, dtype=float)
    lower = bounds[:, 0] + margin
    upper = bounds[:, 1] - margin
    return bool(np.all(point >= lower) and np.all(point <= upper))


layout_rows = {}
for name in YCB_OBJECTS:
    asset = assets[name]
    center_xy = np.asarray(YCB_LAYOUT[name]["pos"][:2], dtype=float)
    origin_z = float(TABLE_TOP_Z + asset.rest_z_offset)
    layout_rows[name] = {
        "center_xy": center_xy,
        "origin_z": origin_z,
        "radius_xy": float(asset.radius_xy),
        "center_on_table": inside_xy(center_xy, table_xy_bounds),
        "footprint_on_table": inside_xy(center_xy, table_xy_bounds, asset.radius_xy),
        "center_in_working_region": inside_xy(center_xy, working_xy_bounds),
    }

pairwise_margins = {}
for left, right in combinations(YCB_OBJECTS, 2):
    distance = np.linalg.norm(layout_rows[left]["center_xy"] - layout_rows[right]["center_xy"])
    required = assets[left].radius_xy + assets[right].radius_xy
    pairwise_margins[(left, right)] = float(distance - required)

layout_checks = {
    "bounds_finite": np.isfinite(table_xy_bounds).all() and np.isfinite(working_xy_bounds).all(),
    "derived_values_finite": all(
        np.isfinite([row["origin_z"], row["radius_xy"]]).all()
        for row in layout_rows.values()
    ),
    "centers_on_table": all(row["center_on_table"] for row in layout_rows.values()),
    "footprints_on_table": all(row["footprint_on_table"] for row in layout_rows.values()),
    "centers_in_working_region": all(
        row["center_in_working_region"] for row in layout_rows.values()
    ),
    "positive_pairwise_margins": all(margin > 0.0 for margin in pairwise_margins.values()),
}

print("table xy bounds:", table_xy_bounds.tolist())
print("working xy bounds:", working_xy_bounds.tolist())
for name, row in layout_rows.items():
    print(
        f"{name:>11}  xy={row['center_xy'].round(3).tolist()}  "
        f"origin_z={row['origin_z']:.6f}  radius={row['radius_xy']:.6f}  "
        f"table={row['footprint_on_table']}  working={row['center_in_working_region']}"
    )
for pair, margin in pairwise_margins.items():
    print(f"pairwise margin {pair[0]} ↔ {pair[1]}: {margin:.6f} m")
for name, passed in layout_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(layout_checks.values()):
    raise AssertionError(layout_checks)


## Build one reusable task scene

The shared builder is the assembly source of truth. It declares the Plane, five fixed table parts, four dynamic YCB objects, Franka, and the requested cameras before one `build()`. After build it configures the robot and attaches the wrist camera. The returned `SceneBundle` lets later code address components by role instead of entity order.


In [ ]:
bundle = build_scene(
    show_viewer=False,
    n_envs=1,
    add_world_cam=render_enabled,
    add_wrist_cam=render_enabled,
    add_video_cam=False,
    draw_world_frame=False,
    scene_dr=None,
)
initial_qpos = to_numpy(bundle.franka.get_qpos()).astype(float)

build_checks = {
    "five_table_entities": len(bundle.table) == 5,
    "four_named_ycb_entities": set(bundle.ycb) == set(YCB_OBJECTS),
    "unbatched_franka_qpos": initial_qpos.shape == (9,),
    "franka_qpos_finite": np.isfinite(initial_qpos).all(),
    "requested_camera_pair": (bundle.world_cam is not None) == render_enabled
    and (bundle.wrist_cam is not None) == render_enabled,
    "no_video_camera": bundle.video_cam is None,
}
for name, passed in build_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(build_checks.values()):
    raise AssertionError(build_checks)

print("table entities:", len(bundle.table))
print("YCB handles:", sorted(bundle.ycb))
print("Franka qpos shape:", initial_qpos.shape)
print("cameras:", "world + wrist" if render_enabled else "SKIP — not created")


## Hold, settle, and measure

A successful build is only the start of runtime evidence. For 60 physics steps, keep sending the configured Franka home target, advance the Scene, and synchronize the attached camera. Then compare measured qpos with home q and inspect each object's world-frame origin and AABB. The AABB bottom, rather than the mesh origin, is the useful tabletop-support measurement.


In [ ]:
SETTLE_STEPS = 60
HOME_Q_TOLERANCE = 0.02
TABLE_SUPPORT_TOLERANCE = 0.005
XY_DRIFT_TOLERANCE = 0.01
home_qpos = np.asarray(FRANKA_QPOS, dtype=float)

for _ in range(SETTLE_STEPS):
    bundle.franka.control_dofs_position(home_qpos)
    bundle.scene.step()
    bundle.update_wrist_cam()

settled_qpos = to_numpy(bundle.franka.get_qpos()).astype(float)
home_q_error = float(np.max(np.abs(settled_qpos - home_qpos)))
object_state = {}
for name, entity in bundle.ycb.items():
    position = to_numpy(entity.get_pos()).astype(float)
    aabb = to_numpy(entity.get_AABB()).astype(float)
    if position.shape != (3,) or aabb.shape != (2, 3):
        raise AssertionError(f"{name}: unexpected position/AABB shapes {position.shape}, {aabb.shape}")
    configured_xy = np.asarray(YCB_LAYOUT[name]["pos"][:2], dtype=float)
    object_state[name] = {
        "position": position,
        "aabb": aabb,
        "support_error": float(abs(aabb[0, 2] - TABLE_TOP_Z)),
        "xy_drift": float(np.linalg.norm(position[:2] - configured_xy)),
    }

state_checks = {
    "settled_qpos_shape": settled_qpos.shape == (9,),
    "settled_qpos_finite": np.isfinite(settled_qpos).all(),
    "home_q_error": home_q_error < HOME_Q_TOLERANCE,
    "object_state_finite": all(
        np.isfinite(row["position"]).all() and np.isfinite(row["aabb"]).all()
        for row in object_state.values()
    ),
    "tabletop_support": all(
        row["support_error"] < TABLE_SUPPORT_TOLERANCE for row in object_state.values()
    ),
    "xy_drift": all(row["xy_drift"] < XY_DRIFT_TOLERANCE for row in object_state.values()),
}

print(f"Franka max |q - home|: {home_q_error:.6f} rad")
for name, row in object_state.items():
    print(
        f"{name:>11}  pos={row['position'].round(6).tolist()}  "
        f"AABB bottom={row['aabb'][0, 2]:.6f}  "
        f"support error={row['support_error']:.6f}  xy drift={row['xy_drift']:.6f}"
    )
for name, passed in state_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(state_checks.values()):
    raise AssertionError(state_checks)


## Compare world and wrist observations

When rendering is enabled, both cameras were declared before build and the wrist camera was synchronized after every step. The fixed world view should expose the global table–robot–object arrangement; the hand-attached wrist view should expose local task geometry. Validate the actual RGB/depth arrays before interpreting either image. With rendering disabled, this branch confirms that no cameras were created and reports an explicit `SKIP`.


In [ ]:
camera_status = "SKIP — ROBO_GENESIS_RENDER=0; world/wrist cameras were not created"
camera_checks = {
    "cameras_absent_when_disabled": not render_enabled
    and bundle.world_cam is None
    and bundle.wrist_cam is None,
}
camera_path_ok = all(camera_checks.values())
camera_frames = {}

if render_enabled:
    rendered = bundle.render(rgb=True, depth=True)
    camera_checks = {"camera_keys": set(rendered) == {"world", "wrist"}}
    expected_resolutions = {"world": WORLD_CAM_RES, "wrist": WRIST_CAM_RES}
    for name in ("world", "wrist"):
        rgb, depth, _, _ = rendered[name]
        rgb = to_numpy(rgb)
        depth = to_numpy(depth)
        width, height = expected_resolutions[name]
        camera_frames[name] = {"rgb": rgb, "depth": depth}
        camera_checks.update(
            {
                f"{name}_rgb_shape": rgb.shape == (height, width, 3),
                f"{name}_depth_shape": depth.shape == (height, width),
                f"{name}_rgb_dtype": rgb.dtype == np.uint8,
                f"{name}_depth_float": np.issubdtype(depth.dtype, np.floating),
                f"{name}_pixels_finite": np.isfinite(rgb).all() and np.isfinite(depth).all(),
                f"{name}_rgb_variation": bool(np.std(rgb) > 0.0),
                f"{name}_positive_depth": bool(np.count_nonzero(depth > 0.0) > 0),
            }
        )
    camera_path_ok = all(camera_checks.values())
    if not camera_path_ok:
        raise AssertionError(camera_checks)
    camera_status = "PASSED — world/wrist RGB and depth captured"

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for axis, name in zip(axes, ("world", "wrist")):
        axis.imshow(camera_frames[name]["rgb"])
        axis.set_title(f"{name.title()} camera — RGB")
        axis.axis("off")
        positive_depth = camera_frames[name]["depth"][camera_frames[name]["depth"] > 0.0]
        print(
            f"{name}: RGB {camera_frames[name]['rgb'].shape}, "
            f"depth range [{positive_depth.min():.4f}, {positive_depth.max():.4f}] m"
        )
    figure.tight_layout()
    plt.show()

if not camera_path_ok:
    raise AssertionError(camera_checks)
print(camera_status)


## Exercise: propose one new banana xy

Change only `BANANA_CANDIDATE_XY`. Before running the cell, predict whether the new center lies in the course working region, keeps the complete conservative footprint on the tabletop, and has positive separation from the lemon, plum, and bowl. This calculation does not rebuild or mutate the current Scene.


In [ ]:
BANANA_CANDIDATE_XY = np.array([0.30, 0.18], dtype=float)
banana_radius = assets["011_banana"].radius_xy
candidate_margins = {}
for other_name in YCB_OBJECTS:
    if other_name == "011_banana":
        continue
    other_xy = np.asarray(YCB_LAYOUT[other_name]["pos"][:2], dtype=float)
    candidate_margins[other_name] = float(
        np.linalg.norm(BANANA_CANDIDATE_XY - other_xy)
        - banana_radius
        - assets[other_name].radius_xy
    )

candidate_checks = {
    "center_in_working_region": inside_xy(BANANA_CANDIDATE_XY, working_xy_bounds),
    "footprint_on_table": inside_xy(BANANA_CANDIDATE_XY, table_xy_bounds, banana_radius),
    "positive_pairwise_margins": all(margin > 0.0 for margin in candidate_margins.values()),
}
print("candidate xy:", BANANA_CANDIDATE_XY.tolist())
for other_name, margin in candidate_margins.items():
    print(f"margin to {other_name}: {margin:.6f} m")
for name, passed in candidate_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
print("candidate feasible:", all(candidate_checks.values()))


## Checkpoint

You have now connected four kinds of evidence: prepared asset records, a checked spatial configuration, named runtime entities after settling, and optional observations from complementary camera poses. L08 can consume this same `SceneBundle` and add task targets and success criteria; it does not need a second scene definition.


In [ ]:
final_checks = {
    "runtime_contract": environment["genesis_world"] == "1.3.3"
    and actual_backend in {"cpu", "amdgpu"}
    and (backend_mode != "cpu" or actual_backend == "cpu"),
    "asset_readiness": all(asset_checks.values()),
    "layout_geometry": all(layout_checks.values()),
    "scene_bundle": all(build_checks.values()),
    "settled_state": all(state_checks.values()),
    "camera_branch": camera_path_ok,
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError("L07 checks failed: " + ", ".join(failed))

print("camera:", camera_status)
print("L07 CHECK: PASSED")
